In [13]:
"""
load_csv_to_mysql.py
---------------------
Loads CSV files into MySQL tables in FK-safe order:

    Category  ->  Product  ->  Customer_Details  ->  Order_Details

Handles, automatically, per table:
  - Missing/unmapped columns (warns, skips)
  - Bad / unparseable dates (logs the offending values, then drops those rows
    if the column is NOT NULL in the DB, or fills a default if you configure one)
  - Rows with NULL in a NOT NULL column (logged to a CSV for review, then dropped)
  - Duplicate primary keys within the same CSV (keeps first, logs the rest)
  - Orphan foreign keys (customer_id / product_id / category_id not present in
    the parent table) -- logged and dropped BEFORE insert, so one bad row can't
    poison an entire chunk
  - Per-chunk insert failures: retries row-by-row so a single bad row doesn't
    lose the other 99 in its chunk
  - A full run summary at the end (rows read / cleaned / inserted / skipped)

Requirements:
    pip install pandas mysql-connector-python sqlalchemy
"""

import sys
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import IntegrityError, SQLAlchemyError

# ------------------------------------------------------------------
# 1. DATABASE CONNECTION -- EDIT THESE VALUES
# ------------------------------------------------------------------
DB_USER = "root"
DB_PASSWORD = "root123"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "farmora"

engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# ------------------------------------------------------------------
# 2. CSV FILE LOCATIONS -- EDIT THESE PATHS
# ------------------------------------------------------------------
CSV_FILES = {
    "Category": "categories.csv",
    "Product": "sales_products.csv",
    "Customer_Details": "Customer.csv",
    "Order_Details": "sales_date.csv",
}

# ------------------------------------------------------------------
# 3. COLUMN MAPPING (CSV header -> DB column)
# ------------------------------------------------------------------
COLUMN_MAP = {
    "Category": {
        "category_id": "category_id",
        "category_name": "category_name",
        "description": "description",
    },
    "Product": {
        "product_id": "product_id",
        "category_id": "category_id",
        "product_name": "product_name",
        "price": "price",
        "manufacture_date": "manufacture_date",
        "expiry_date": "expiry_date",
        "quantity": "quantity",
        "discount": "discount",
    },
    "Customer_Details": {
        "customer_id": "customer_id",
        "FirstName": "FirstName",
        "MiddleName": "MiddleName",
        "LastName": "LastName",
        "Address": "Address",
        "customer_name": "customer_name",
        "email_id": "email_id",
        "password": "password",
    },
    "Order_Details": {
        "order_id": "order_id",
        "customer_id": "customer_id",
        "product_id": "product_id",
        "product_count": "product_count",
        "product_price": "product_price",
        "product_discount": "product_discount",
        "price_after_discount": "price_after_discount",
        "order_date": "order_date",
    },
}

# Columns that must be parsed as dates
DATE_COLUMNS = {
    "Product": ["manufacture_date", "expiry_date"],
    "Order_Details": ["order_date"],
}

# Primary key column per table (used for duplicate detection within the CSV)
PRIMARY_KEY = {
    "Category": "category_id",
    "Product": "product_id",
    "Customer_Details": "customer_id",
    "Order_Details": "order_id",
}

# Columns that are NOT NULL in the DB schema -- rows with a null here get
# logged and dropped rather than crashing the whole chunk.
NOT_NULL_COLUMNS = {
    "Category": ["category_id", "category_name"],
    "Product": ["product_id", "category_id", "product_name", "price"],
    "Customer_Details": ["customer_id", "customer_name", "email_id"],
    "Order_Details": [
        "order_id", "customer_id", "product_id",
        "product_count", "product_price", "order_date",
    ],
}

# Foreign keys to validate before insert: {table: {fk_column: (parent_table, parent_pk_column)}}
FOREIGN_KEYS = {
    "Product": {"category_id": ("Category", "category_id")},
    "Order_Details": {
        "customer_id": ("Customer_Details", "customer_id"),
        "product_id": ("Product", "product_id"),
    },
}

TABLE_ORDER = ["Category", "Product", "Customer_Details", "Order_Details"]
CHUNK_SIZE = 500
REJECTS_DIR = "rejected_rows"  # bad rows get written here as <table>_rejected.csv


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def get_existing_pks(table_name: str, pk_col: str) -> set:
    """Fetch the set of primary key values already in the parent table (DB)."""
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT `{pk_col}` FROM `{table_name}`"))
            return {row[0] for row in result}
    except SQLAlchemyError:
        # Table might not exist yet / be empty -- treat as no existing rows.
        return set()


def save_rejects(table_name: str, df_rejects: pd.DataFrame, reason: str):
    if df_rejects.empty:
        return
    import os
    os.makedirs(REJECTS_DIR, exist_ok=True)
    out_path = f"{REJECTS_DIR}/{table_name}_rejected.csv"
    df_out = df_rejects.copy()
    df_out["__reject_reason"] = reason
    # Append if file already has rejects from an earlier stage in this run
    write_header = not os.path.exists(out_path)
    df_out.to_csv(out_path, mode="a", header=write_header, index=False)
    print(f"  -> {len(df_rejects)} rows rejected ({reason}). Logged to {out_path}")


# ------------------------------------------------------------------
# Core loader
# ------------------------------------------------------------------
def load_table(table_name: str):
    csv_path = CSV_FILES[table_name]
    col_map = COLUMN_MAP[table_name]

    print(f"\n=== Loading {table_name} from {csv_path} ===")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"  ERROR: file not found: {csv_path}. Skipping table.")
        return {"read": 0, "inserted": 0, "rejected": 0}

    total_read = len(df)

    # --- 1. Column mapping ---
    missing = [c for c in col_map if c not in df.columns]
    if missing:
        print(f"  WARNING: columns missing in CSV, skipping them: {missing}")

    usable_cols = [c for c in col_map if c in df.columns]
    df = df[usable_cols].rename(columns={c: col_map[c] for c in usable_cols})

    # --- 2. Parse date columns, log unparseable values ---
    for date_col in DATE_COLUMNS.get(table_name, []):
        if date_col not in df.columns:
            continue
        raw = df[date_col]
        parsed = pd.to_datetime(raw, errors="coerce", dayfirst=False)
        bad_mask = parsed.isna() & raw.notna() & (raw.astype(str).str.strip() != "")
        if bad_mask.any():
            print(f"  WARNING: {bad_mask.sum()} unparseable '{date_col}' values, e.g.:")
            print("   ", raw[bad_mask].unique()[:5])
        df[date_col] = parsed.dt.date

    # --- 3. Drop fully empty rows ---
    df = df.dropna(how="all")

    # --- 4. Drop / log rows violating NOT NULL columns ---
    required_cols = [c for c in NOT_NULL_COLUMNS.get(table_name, []) if c in df.columns]
    if required_cols:
        null_mask = df[required_cols].isna().any(axis=1)
        if null_mask.any():
            save_rejects(table_name, df[null_mask], f"null in required column {required_cols}")
            df = df[~null_mask]

    # --- 5. Drop duplicate primary keys within this CSV ---
    pk_col = PRIMARY_KEY.get(table_name)
    if pk_col and pk_col in df.columns:
        dup_mask = df.duplicated(subset=[pk_col], keep="first")
        if dup_mask.any():
            save_rejects(table_name, df[dup_mask], f"duplicate {pk_col} within CSV")
            df = df[~dup_mask]

        # Drop rows whose PK already exists in the DB (avoids IntegrityError on rerun)
        existing_pks = get_existing_pks(table_name, pk_col)
        if existing_pks:
            already_mask = df[pk_col].isin(existing_pks)
            if already_mask.any():
                print(f"  Skipping {already_mask.sum()} rows already present in DB (by {pk_col}).")
                df = df[~already_mask]

    # --- 6. Validate foreign keys against parent tables ---
    for fk_col, (parent_table, parent_pk) in FOREIGN_KEYS.get(table_name, {}).items():
        if fk_col not in df.columns:
            continue
        valid_pks = get_existing_pks(parent_table, parent_pk)
        # Also include any PKs we're inserting for the parent in THIS run, if
        # the parent table was already loaded earlier in TABLE_ORDER -- the
        # get_existing_pks call above already picks these up since they were
        # committed to the DB before we get here (tables load in FK order).
        orphan_mask = ~df[fk_col].isin(valid_pks)
        if orphan_mask.any():
            save_rejects(table_name, df[orphan_mask], f"orphan FK {fk_col} -> {parent_table}.{parent_pk}")
            df = df[~orphan_mask]

    print(f"  {len(df)} / {total_read} rows clean and ready to insert.")

    if df.empty:
        return {"read": total_read, "inserted": 0, "rejected": total_read}

    # --- 7. Insert in chunks, falling back to row-by-row on chunk failure ---
    inserted = 0
    rejected_on_insert = 0
    for start in range(0, len(df), CHUNK_SIZE):
        chunk = df.iloc[start:start + CHUNK_SIZE]
        try:
            chunk.to_sql(name=table_name, con=engine, if_exists="append", index=False)
            inserted += len(chunk)
        except SQLAlchemyError as e:
            print(f"  Chunk [{start}:{start+len(chunk)}] failed as a batch "
                  f"({type(e).__name__}); retrying row-by-row...")
            for _, row in chunk.iterrows():
                try:
                    pd.DataFrame([row]).to_sql(
                        name=table_name, con=engine, if_exists="append", index=False
                    )
                    inserted += 1
                except SQLAlchemyError as row_err:
                    rejected_on_insert += 1
                    save_rejects(
                        table_name,
                        pd.DataFrame([row]),
                        f"DB rejected on insert: {row_err.__class__.__name__}",
                    )

    print(f"  Inserted {inserted} rows into {table_name}. "
          f"({rejected_on_insert} rejected at insert time)")

    return {
        "read": total_read,
        "inserted": inserted,
        "rejected": total_read - inserted,
    }


def main():
    summary = {}
    for table in TABLE_ORDER:
        summary[table] = load_table(table)

    print("\n" + "=" * 50)
    print("RUN SUMMARY")
    print("=" * 50)
    for table, stats in summary.items():
        print(f"  {table:<18} read={stats['read']:<8} "
              f"inserted={stats['inserted']:<8} rejected={stats['rejected']}")
    print(f"\nRejected rows (if any) are saved per-table in ./{REJECTS_DIR}/")
    print("Done.")


if __name__ == "__main__":
    main()


=== Loading Category from categories.csv ===
  10 / 10 rows clean and ready to insert.
  Inserted 10 rows into Category. (0 rejected at insert time)

=== Loading Product from sales_products.csv ===
  100 / 100 rows clean and ready to insert.
  Inserted 100 rows into Product. (0 rejected at insert time)

=== Loading Customer_Details from Customer.csv ===
  98759 / 98759 rows clean and ready to insert.
  Inserted 98759 rows into Customer_Details. (0 rejected at insert time)

=== Loading Order_Details from sales_date.csv ===
  -> 15029 rows rejected (null in required column ['order_id', 'customer_id', 'product_id', 'product_count', 'product_price', 'order_date']). Logged to rejected_rows/Order_Details_rejected.csv
  1494806 / 1509835 rows clean and ready to insert.
  Inserted 1494806 rows into Order_Details. (0 rejected at insert time)

RUN SUMMARY
  Category           read=10       inserted=10       rejected=0
  Product            read=100      inserted=100      rejected=0
  Customer_De

In [9]:
!pip install mysql

In [6]:
pip install mysql-connector-python pandas sqlalchemy

   ---------------------------------------- 0.0/18.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.8/18.2 MB 14.0 MB/s eta 0:00:02
   ---------- ----------------------------- 4.7/18.2 MB 13.6 MB/s eta 0:00:01
   -------------------- ------------------- 9.2/18.2 MB 16.1 MB/s eta 0:00:01
   ------------------------------- -------- 14.4/18.2 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------  18.1/18.2 MB 19.4 MB/s eta 0:00:01
   ---------------------------------------- 18.2/18.2 MB 17.7 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.
